In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn
import numpy as np
import json
from dotenv import load_dotenv
from sklearn import KNearestClassifier

# pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)  
# pd.set_option('display.max_colwidth', None)
data = pd.read_json('appraisals_dataset.json')


**Get all subjects for each appraisal**

In [ ]:
all_subjects = []
for appraisal in data['appraisals']:
    all_subjects.append(appraisal['subject'])
    
df_subjects = pd.DataFrame(all_subjects)
df_subjects.info()

**Get all properties**

In [ ]:
all_properties = []
for appraisal in data['appraisals']:
    for property in appraisal['properties']:
        all_properties.append(property)

df_properties = pd.DataFrame(all_properties)
df_properties.info()


**Select numerical property features** 

In [ ]:
df_properties.describe()
df_subjects.describe()

In [ ]:
df_properties[['gla', 'room_count', 'full_baths', 'half_baths', 'bedrooms', 'lot_size_sf', 'year_built']].info()

In [ ]:
df_properties.isnull().sum()

**Drop property columns with too many missing data**

In [ ]:
df_properties.drop(['main_level_finished_area', 'bg_fin_area', 'upper_lvl_fin_area', 'public_remarks'], axis=1, inplace=True)

**Select relevant columns common to both subject and properties**

In [ ]:
print(f"unique columns in subjects: {df_subjects.columns.__len__()}")
print(f"unique columns in properties: {df_properties.columns.__len__()}")


In [ ]:
props_selected = df_properties[['bedrooms', 'gla', 'year_built', 'structure_type', 'lot_size_sf', 'basement', 'heating', 'cooling', 'style']]
subs_selected = df_subjects[['num_beds', 'gla', 'year_built', 'structure_type', 'lot_size_sf', 'basement', 'heating','cooling', 'style']]

In [ ]:
#set missing bedroom values with median
props_selected['bedrooms']
median = props_selected['bedrooms'].median()
props_selected.fillna({'bedrooms': median}, inplace=True)


In [ ]:
#convert number of beds for selected subjects to floats
subs_selected['num_beds'] = pd.to_numeric(subs_selected['num_beds'], errors='coerce')

In [ ]:
#find out how many num_beds were not able to be converted
subs_selected['num_beds'].info()

#fill null values of num_beds with median
subs_selected['num_beds'] = subs_selected['num_beds'].fillna(subs_selected['num_beds'].median())


In [ ]:
subs_selected['gla'] = df_subjects['gla']

In [ ]:
#clean gla values for subs_selected
import re
def parse_areas(gla_value):
    if isinstance(gla_value, str):
        # Remove commas
        gla_value = gla_value.replace(',', '').lower()
        # Check for SqM or sqm (case-insensitive)
        if 'sqm' in gla_value:
            num = float(re.sub('[^0-9.]+', '', gla_value))
            # 1 SqM = 10.7639 SqFt
            return num * 10.7639
        elif 'acres' in gla_value:
            num = float(re.sub('[^0-9.]+', '', gla_value))
        elif 'sqft' in gla_value:
            value = re.sub('[^0-9.]+', '', gla_value)
            if value:
                return float(value)
            return np.nan
        else:
            return np.nan

subs_selected['gla'] = subs_selected['gla'].apply(parse_areas)

In [ ]:
#clean gla values for props_selected(replacing missing values with median)
median = props_selected['gla'].median()
props_selected.fillna({'gla': median}, inplace=True)

In [ ]:
#drop year_built for both dataframes for now
subs_selected.drop(columns=['year_built'], inplace=True)
props_selected.drop(columns=['year_built'], inplace=True)


In [ ]:
#clean lot_size_sf for both dataframes
props_selected.fillna({'lot_size_sf': props_selected['lot_size_sf'].median()}, inplace=True)


In [ ]:
#clean subs_selected lot sizes
subs_selected['lot_size_sf'].unique()

In [ ]:
subs_selected['lot_size_sf']=subs_selected['lot_size_sf'].apply(parse_areas)
subs_selected['lot_size_sf'].info()

In [ ]:
#remove null lot_sizes
subs_selected.fillna({'lot_size_sf': subs_selected['lot_size_sf'].median()}, inplace=True)

In [ ]:
props_selected['heating'].describe()

In [ ]:
#standardize heating entries for all properties
def categorize_heating(heating_str):
    if pd.isna(heating_str) or heating_str == '' or heating_str == 'None' or heating_str == 'Unknown ' or heating_str == 'See Remarks ':
        return 'Unknown'
    
    heating_lower = str(heating_str).lower().strip()
    
    # Forced Air Systems
    if 'forced air' in heating_lower or 'furnace' in heating_lower:
        return 'Forced Air'
    
    # Heat Pump Systems
    elif 'heat pump' in heating_lower:
        return 'Heat Pump'
    
    # Electric Systems (Baseboard, ETS)
    elif 'baseboard' in heating_lower or 'electric' in heating_lower or 'ets' in heating_lower:
        return 'Electric'
    
    # Hydronic Systems (Hot Water, Boiler, Radiant, Steam)
    elif any(x in heating_lower for x in ['hot water', 'boiler', 'radiant', 'steam', 'water']):
        return 'Hydronic'
    
    # Alternative/Supplementary (Stove, Fireplace)
    elif any(x in heating_lower for x in ['stove', 'fireplace', 'pellet']):
        return 'Alternative'
    
    # Everything else
    else:
        return 'Other'

props_selected['heating'] = props_selected['heating'].apply(categorize_heating)

In [ ]:
subs_selected['heating'] = subs_selected['heating'].apply(categorize_heating)

In [ ]:
#standard the basement area categories
def categorize_basement(basement_str):
    if pd.isna(basement_str) or basement_str == '' or basement_str == 'None' or basement_str == 'Unknown ' or basement_str == 'See Remarks ' or basement_str == 'None (No Basement)':
        return 'No Basement'
    
    basement_lower = str(basement_str).lower().strip()
    
    # Fully Finished
    if any(x in basement_lower for x in ['finished', 'fully developed', 'fully finished']):
        # With walkout
        if any(x in basement_lower for x in ['walk-out', 'walkout', 'w/o']):
            return 'Finished Walkout'
        return 'Finished'
    
    # Partially Finished
    elif any(x in basement_lower for x in ['part fin', 'partially', 'part bsmt']):
        # With walkout
        if any(x in basement_lower for x in ['walk-out', 'walkout', 'w/o']):
            return 'Partially Finished Walkout'
        return 'Partially Finished'
    
    # Unfinished
    elif 'unfinished' in basement_lower or 'undeveloped' in basement_lower:
        # With walkout
        if any(x in basement_lower for x in ['walk-out', 'walkout', 'w/o']):
            return 'Unfinished Walkout'
        return 'Unfinished'
    
    # Crawl Space
    elif 'crawl space' in basement_lower:
        return 'Crawl Space'
    
    # Everything else
    else:
        return 'Other'

props_selected['basement'] = props_selected['basement'].apply(categorize_basement)

In [ ]:
#categorize subjects 
subs_selected['basement'] = subs_selected['basement'].apply(categorize_basement).info()

In [ ]:
def categorize_cooling(cooling_str):
    if pd.isna(cooling_str) or cooling_str == '' or cooling_str == 'None' or cooling_str == 'Unknown ' or cooling_str == 'See Remarks ' or cooling_str == 'Other ':
        return 'None'
    
    cooling_lower = str(cooling_str).lower().strip()
    
    # Central Air
    if 'central air' in cooling_lower:
        return 'Central Air'
    
    # Heat Pump (includes both ducted and ductless)
    elif 'heat pump' in cooling_lower or 'ductless' in cooling_lower or 'ducted' in cooling_lower:
        return 'Heat Pump'
    
    # Wall/Window Units
    elif any(x in cooling_lower for x in ['wall unit', 'window unit']):
        return 'Wall/Window Units'
    
    # Everything else
    else:
        return 'Other'
    
props_selected['cooling'] = props_selected['cooling'].apply(categorize_cooling)



In [ ]:
#clean the subjects cooling column
subs_selected['cooling'] = props_selected['cooling'].apply(categorize_cooling)

**One-Hot Encond Categorical Features for clustering**

In [ ]:
#drop style and structure type columns since it won't be used in clustering
subs_selected.drop(columns='style', inplace=True)
props_selected.drop(columns='style', inplace=True)

props_selected.drop(columns='structure_type', inplace=True)
subs_selected.drop(columns='structure_type', inplace=True)


In [ ]:
#select columns for one-hot encoding
encode_cols = ['basement', 'heating', 'cooling']
encoded_props = pd.get_dummies(props_selected, columns=encode_cols, dtype=int)
encoded_subjects = pd.get_dummies(subs_selected, columns=encode_cols, dtype=int)

In [ ]:
#initialize model


In [ ]:
#sacle numerical property features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

props_scaled = encoded_props.copy()
numerical_features = ['bedrooms', 'gla', 'lot_size_sf']
props_scaled[numerical_features] =  scaler.fit_transform(encoded_props[numerical_features])



In [ ]:
#rename subs_selected columns to match properties 
subs_selected = subs_selected.rename(columns={'num_beds': 'bedrooms'})

In [ ]:
#scale numerical features of subjects 
numerical_features_subjects = ['bedrooms','gla','lot_size_sf']

subs_scaled = encoded_subjects.copy()


subs_scaled[numerical_features_subjects]  = scaler.transform(encoded_subjects[numerical_features_subjects])

In [ ]:
subs_scaled

In [401]:
#traning the nearest neighbor model
from sklearn.neighbors import NearestNeighbors

n_neighbors = 10
nn = NearestNeighbors(n_neighbors=n_neighbors, metric='euclidean')


In [ ]:
props_scaled_correction = props_scaled.copy()

In [ ]:
#drop columns that are only in properties
drop_cols = [col for col in props_scaled.columns if col not in subs_scaled.columns]
props_scaled_correction.drop(columns=drop_cols, inplace=True)

In [385]:
#retrain with new data
props_scaled_correction.columns

Index(['bedrooms', 'gla', 'lot_size_sf', 'basement_Finished',
       'basement_Finished Walkout', 'basement_No Basement', 'basement_Other',
       'heating_Electric', 'heating_Forced Air', 'heating_Heat Pump',
       'heating_Hydronic', 'heating_Other', 'heating_Unknown',
       'cooling_Central Air', 'cooling_None', 'cooling_Wall/Window Units'],
      dtype='object')

In [386]:
subs_scaled.columns

Index(['bedrooms', 'gla', 'lot_size_sf', 'basement_Finished',
       'basement_Finished Walkout', 'basement_No Basement', 'basement_Other',
       'heating_Electric', 'heating_Forced Air', 'heating_Heat Pump',
       'heating_Hydronic', 'heating_Other', 'heating_Unknown',
       'cooling_Central Air', 'cooling_None', 'cooling_Wall/Window Units'],
      dtype='object')

In [402]:
nn.fit(props_scaled_correction)

NearestNeighbors(metric='euclidean', n_neighbors=10)

In [403]:
distances, indices = nn.kneighbors(subs_scaled)

In [404]:
print("\nDistances to nearest neighbors:")
print(distances[0])


Distances to nearest neighbors:
[0.01080639 0.01917595 0.02795073 0.02818981 0.03680231 0.04199329
 0.04199329 0.0426848  0.0426848  0.04474723]


In [405]:
print("\nIndices of nearest neighbors:")
print(indices[0])



Indices of nearest neighbors:
[ 257 8641 8566 8513  224 1181 6487  419 7305 4743]


In [ ]:
df_properties.iloc[]

id                                                124983
address                            218 Doon Mills Drive 
bedrooms                                             3.0
gla                                               1669.0
city                                          Kitchener 
province                                         Ontario
postal_code                                      N2P 2R9
property_sub_type                Single Family Residence
structure_type       Single Family Residence, Two Story 
style                                         Two Story 
levels                                        Two Story 
room_count                                          11.0
full_baths                                           1.0
half_baths                                           1.0
lot_size_sf                                       5520.0
year_built                                        2003.0
roof                                      Asphalt Shing 
basement                       

In [407]:
df_subjects.iloc[0]

address                      142-950 Oakview Ave Kingston ON K7M 6W8
subject_city_province_zip                         "Twin Oak Meadows"
effective_date                                           Apr/11/2025
municipality_district                                       Kingston
site_dimensions                                Condo Common Property
lot_size_sf                                                      n/a
units_sq_ft                                                     SqFt
year_built                                                      1976
structure_type                                             Townhouse
roofing                                              Asphalt Shingle
effective_age                                                     25
style                                                       2 Storey
construction                                              Wood Frame
remaining_economic_life                                           50
windows                           